### 1)

Vi skal løse likningen 
\begin{aligned} 
\partial_t u - \nabla \cdot (M \nabla (f(u) - \kappa \Delta u)) = g, \quad \Omega \times (0, T)
\end{aligned}

Ved å bruke forenklingene brukt i tidligere oppgaver får vi
\begin{aligned} 
\partial_t u - \kappa \Delta^2u - \Delta(u^3 - u) = g
\end{aligned}

$\partial_t u$, $\kappa \Delta^2u$ og $\Delta u$ er lineære ledd som skal behandles implisitt, mens $\Delta u^3$ er ikke-lineær og skal behandles eksplisitt. $g$ skal også behandles implisitt.

For å unngå uheldige fortegn splitter vi opp den ikke-lineære delen på følgende måte
\begin{aligned}
\Delta (u^3 -u) = 
\underbrace{a\Delta u}_{f_1(u)} + \underbrace{\Delta u^3 -(1+a) \Delta u}_{f_2(u)}.
\end{aligned}

der vi behandler $f_1(u)$ implisitt og $f_2(u)$ eksplisitt. 


Vi samler leddene som skal behandles implisitt på venstre side og leddene som skal behandles eksplisitt på høyre side og får
$$
\partial_t u - \kappa \Delta^2u - a \Delta u - g = \Delta u^3 - (1 + a) \Delta u
$$

Dette gir oss følgende IMEX-metode

\begin{aligned}
\dfrac{u^{n+1} - u^n}{\tau} +\kappa \Delta^2 u^{n+1}  - a \Delta u^{n+1} - g^{n+1} &= \Delta \bigl( (u^n)^3 - (1+a) u^n\bigr)
\\
\\
u^{n+1} +\tau (\kappa \Delta^2 u^{n+1}  - a \Delta u^{n+1}) - \tau g^{n+1} &= u^n + \tau \Delta \bigl( (u^n)^3 - (1+a) u^n\bigr)
\end{aligned}

Deretter transformer vi likningen til Fourier-rommet
\begin{aligned}
\widehat{u}^{n+1} +\tau (\kappa \widetilde{\mathbf{k}}^4 u^{n+1}  - a \widetilde{\mathbf{k}}^2 u^{n+1}) - \tau \widehat{g}^{n+1} 
&= u^n + \tau \widetilde{\mathbf{k}}^2 \widehat{(u^n)^3 - (1+a) u^n}
\end{aligned}

Merk at vi beholder $\widehat{(u^n)^3 - (1+a) u^n}$ på denne formen da dette skal løses i det fysiske rom og deretter transformeres. 

Ved å løse for $\widehat{u}^{n+1}$ får vi et uttrykk for den nye løsningen i Fourier-rommet:
\begin{aligned}
\widehat{u}^{n+1} = \frac{\widehat{u}^n + \tau (\widetilde{\mathbf{k}}^2 \widehat{(u^n)^3 - (1+a) u^n} + \widehat g)}
{1 + \tau (\kappa \widetilde{\mathbf{k}}^4 - a \widetilde{\mathbf{k}}^2)}
\end{aligned}

### 2)

In [1]:
import numpy as np
from scipy.fft import fft, ifft, fft2, ifft2, fftfreq, fftshift
import sympy as sp

In [ ]:
def cahn_hilliard_backward_euler(*, 
                                kappa, 
                                X, Y, U0, 
                                t0, T, Nt,
                                g, 
                                alpha=1.5):
    """
    Implements the Cahn-Hilliard equation solver using the backward Euler method 
    with a convex-concave splitting approach.

    Parameters:
    -----------
    kappa : float
        Diffusion coefficient for the biharmonic operator.
    X : ndarray
        2D array representing the x-coordinates of the grid.
    Y : ndarray
        2D array representing the y-coordinates of the grid.
    U0 : ndarray
        Initial condition for the solution.
    t0 : float
        Initial time.
    T : float
        Final time.
    Nt : int
        Number of time steps.
    g : callable or None
        Source term as a function of (X, Y, t). If None, no source term is applied.
    alpha : float, optional
        Convex-concave splitting parameter. Default is 1.5.

    Yields:
    -------
    tuple: A tuple containing the discrete Fourier transform of U at t, and the current time t.

    """

    # Prepare relevant data for Fourier transform
    N, M = U0.shape  # U0 is a NxM matrix, but in python .shape gives them in reverse order
    dx = X[0, 1] - X[0, 0]
    dy = Y[1, 0] - Y[0, 0]
    k_x = (fftfreq(N, d=dx/(2*np.pi)))
    k_y = (fftfreq(M, d=dy/(2*np.pi)))

    KX, KY = np.meshgrid(k_x, k_y, indexing="ij")
    K2 = KX**2 + KY**2
    K4 = K2**2

    # Compute DFT of the initial value
    U_hat = fft2(U0)

    # Add your time-stepping loop here
    # Time stepping 
    t = t0 
    dt = (T-t0)/Nt

    # For convenience when plotting, computing errors, etc., 
    # return the initial solution and initial time.
    yield U_hat, t  

    
    while t < T-dt/2:
        # Compute source term in Fourier space if provided
        if g is not None:
            g_hat = fft2(g(X, Y, t + dt))                           # g skal behandles implisitt, bruker t + dt
        else:
            g_hat = 0
    

        U_lin = (ifft2(U_hat))**3 - (1+alpha)*(ifft2(U_hat))        # Finner lineær del i det fysiske rom
        U_lin_hat = fft2(U_lin)                                     # Finner lineær del i Fourierrommet 

        denominator = 1 + dt*(kappa*K4 - alpha*K2)
        numerator = U_hat + dt*(K2*U_lin_hat + g_hat)
        U_hat = numerator/denominator

        # Update time
        t += dt

        yield U_hat, t